# Anima + RouWei + Nova Anime XL + JANKU — ComfyUI Colab Compare

Сравнительный запуск ComfyUI для 4 Illustrious checkpoints: **Anima Aesthetic v1.1**, **RouWei v0.8 epsilon**, **Nova Anime XL v19**, **JANKU v7.77**. Устанавливает ComfyUI + Manager + Anima-LLLite, скачивает единый Qwen VAE/text-enc и все 4 чекпоинта, запускает Cloudflare Tunnel и очередь compare (10 промптов × 4 модели). Токены из Colab Secrets (`HF_TOKEN`, `CIVITAI_API_TOKEN`).

- Anima (diffusion model): `diffusion_models/anima/anima_aestheticV11.safetensors` + `text_encoders/qwen_3_06b_base.safetensors` + `vae/qwen_image_vae.safetensors` + `model_patches/anima-lllite-any-test-like-v2.safetensors`
- RouWei / Nova / JANKU (SDXL checkpoints): `models/checkpoints/*.safetensors` (VAE baked in)
- Промпты: 10 сложных поз/композиций из `models.json` (gravity_workshop, mirror_train, ... museum_giant), seed 424242


In [ ]:
# @title 1) Tokens (Colab Secrets / environment only)
# Tokens are read from Colab Secrets (panel 🔑) or process environment.
# Handles late "Grant Access": if secret not found immediately we poll up to ~90s
# so user can enable Notebook access ON + Grant Access during first cell.
import os, time

!pip install -q -U huggingface_hub
from huggingface_hub import login

try:
    from google.colab import userdata
except Exception:
    userdata = None


def colab_secret(name: str) -> str:
    """Read a secret from env first, then Colab Secrets."""
    value = os.environ.get(name, "").strip()
    if not value and userdata is not None:
        try:
            raw = userdata.get(name) or ""
        except Exception:
            raw = ""
        if isinstance(raw, dict):
            raw = raw.get("value") or raw.get("token") or next(iter(raw.values()), "")
        value = str(raw).strip()
    return value


HF_TOKEN = colab_secret("HF_TOKEN") or colab_secret("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    print("HF_TOKEN не найден — жду до 90 сек (можешь сейчас нажать Grant Access / включить Notebook access ON)...")
    for _attempt in range(30):
        time.sleep(3)
        HF_TOKEN = colab_secret("HF_TOKEN") or colab_secret("HUGGINGFACE_TOKEN")
        if HF_TOKEN:
            print("✓ HF_TOKEN появился — продолжаю.")
            break
        print(".", end="", flush=True)
    print()
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found.\n"
        "  • Colab: открой 🔑 Secrets слева, добавь HF_TOKEN, включи Notebook access ON, нажми Grant Access, затем Runtime → Rerun.\n"
        "  • Если ты нажал Grant Access только что — токен должен был подхватиться за 90 сек; если не подхватился, перезапусти ячейку.\n"
        "  • Jupyter локально: export HF_TOKEN=... перед стартом."
    )
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF auth: OK")

CIVITAI_API_TOKEN = colab_secret("CIVITAI_API_TOKEN")
if CIVITAI_API_TOKEN:
    os.environ["CIVITAI_API_TOKEN"] = CIVITAI_API_TOKEN
    print("Civitai auth: OK")
else:
    print("Civitai token not set (optional); Civitai downloads may return 403.")

print("Done.")


In [ ]:
# @title Install bundled workflow in ComfyUI
import json
from pathlib import Path
from urllib.request import urlopen

WORKFLOW_URL = "https://raw.githubusercontent.com/ekkonwork/free-comfyui-colab-pack/main/workflows/anima_illustrious_compare/workflow.json"
WORKFLOW_DIR = Path("/content/ComfyUI/user/default/workflows")
WORKFLOW_PATH = WORKFLOW_DIR / "anima_illustrious_compare.json"
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)
workflow = json.loads(urlopen(WORKFLOW_URL, timeout=60).read().decode("utf-8"))
replacements = {

}

def replace_runtime_names(value):
    if isinstance(value, str):
        return replacements.get(value, value)
    if isinstance(value, list):
        return [replace_runtime_names(item) for item in value]
    if isinstance(value, dict):
        return {key: replace_runtime_names(item) for key, item in value.items()}
    return value

workflow = replace_runtime_names(workflow)
WORKFLOW_PATH.write_text(json.dumps(workflow, ensure_ascii=False, indent=2), encoding="utf-8")
print("Bundled workflow installed:", WORKFLOW_PATH)


In [ ]:
# @title 2) Install ComfyUI + Managers + node pack (incl. rgthree)
# Idempotent: safe to rerun. Unified template: swap + aria2 split.
import os
import subprocess
import sys

def run(cmd):
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=True)

# Swap protects small-RAM Colab VMs during dependency resolution.
if not os.path.exists("/swapfile"):
    print("Creating swap (8GB)...")
    try:
        subprocess.run("sudo fallocate -l 8G /swapfile && sudo chmod 600 /swapfile && sudo mkswap /swapfile && sudo swapon /swapfile", shell=True, check=True)
        print("Swap enabled.")
    except Exception as e:
        print(f"Swap creation failed ({e}), continuing.")

# System deps: aria2 + ffmpeg (cloudflared handled in launch cell)
import shutil
if shutil.which("aria2c") is None:
    print("Installing aria2 + ffmpeg...")
    subprocess.run("apt-get -y update -qq", shell=True, check=False)
    subprocess.run("apt-get -y install -qq aria2 ffmpeg || true", shell=True, check=False)
else:
    print("aria2c already present:", shutil.which("aria2c"))

COMFY_ROOT_D = "/content/ComfyUI"
MODEL_ROOT = os.path.join(COMFY_ROOT_D, "models")

if not os.path.exists(COMFY_ROOT_D):
    run("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI")
run(f"pip install -q -r {COMFY_ROOT_D}/requirements.txt")

NODES = {
    "ComfyUI-Manager": "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "ComfyUI-Model-Manager": "https://github.com/hayden-cn/ComfyUI-Model-Manager.git#v2.8.4",
    "ComfyUI-GGUF": "https://github.com/city96/ComfyUI-GGUF.git",
    "ComfyUI-Impact-Pack": "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    "ComfyUI-Impact-Subpack": "https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git",
    "comfyui-kjnodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "rgthree-comfy": "https://github.com/rgthree/rgthree-comfy.git",
    "seedvr2_videoupscaler": "https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git",
    "ComfyUI-Workflow-Models-Downloader": "https://github.com/slahiri/ComfyUI-Workflow-Models-Downloader.git",
    # Anima-specific nodes:
    "ComfyUI-Anima-LLLite": "https://github.com/kohya-ss/ComfyUI-Anima-LLLite.git",
    "comfyui_controlnet_aux": "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    "comfyui-lora-manager": "https://github.com/willmiao/ComfyUI-Lora-Manager.git",
    "was-node-suite-comfyui": "https://github.com/WASasquatch/was-node-suite-comfyui.git",
    "ComfyUI-Image-Saver": "https://github.com/alexopus/ComfyUI-Image-Saver.git",
}
for folder, repo in NODES.items():
    target = os.path.join(COMFY_ROOT_D, "custom_nodes", folder)
    if not os.path.exists(target):
        repo_url, _, ref = repo.partition("#")
        clone_cmd = ["git", "clone", "--depth", "1"]
        if ref:
            clone_cmd += ["--branch", ref]
        clone_cmd += [repo_url, target]
        print("+", " ".join(clone_cmd))
        subprocess.run(clone_cmd, check=True)
    req = os.path.join(target, "requirements.txt")
    if os.path.exists(req):
        run(f"pip install -q -r {req}")
print("ComfyUI and node set are ready.")


In [ ]:
# @title 3) Download checkpoints and Anima dependencies (Anima + 3 Illustrious)
import hashlib, json, requests
from pathlib import Path
def verify_sha256(path, expected):
    if not expected: return True
    import hashlib
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda: f.read(4*1024*1024), b''):
            h.update(chunk)
    got=h.hexdigest().upper()
    exp=expected.upper()
    if got!=exp:
        raise RuntimeError(f"SHA256 mismatch {{path}}: expected {{exp}}, got {{got}}")
    print(f"SHA256 OK: {path}")
    return True

MANIFEST = {
  "seed": 424242,
  "prompt": "masterpiece, best quality, highres, 1girl, solo, original anime character, short auburn hair, navy raincoat, standing at a tram stop at dusk, wet pavement reflecting warm shop lights, detailed city background, cinematic composition",
  "negative_prompt": "worst quality, low quality, blurry, bad anatomy, bad hands, extra fingers, watermark, text, signature",
  "prompts": [
    {
      "id": "gravity_workshop",
      "prompt": "masterpiece, best quality, highres, 1girl, solo, adult woman, upside-down suspended pose inside a zero-gravity clockwork workshop, one hand reaching toward a floating brass astrolabe, asymmetrical cobalt flight suit with transparent panels, loose ribbons and tools orbiting around her, impossible perspective, dramatic rim light, intricate mechanical background, cinematic anime illustration"
    },
    {
      "id": "mirror_train",
      "prompt": "masterpiece, best quality, highres, 1girl, solo, adult woman balancing in a one-legged dancer pose on the roof of a moving night train, long mirrored coat over sculptural white clothing, windblown silver hair, neon railway reflected in rain, multiple reflections in broken glass panels, wide-angle composition, surreal city lights, highly detailed anime illustration"
    },
    {
      "id": "desert_observatory",
      "prompt": "masterpiece, best quality, highres, 1girl, solo, adult woman kneeling sideways on a giant observatory telescope, looking over her shoulder while adjusting a star map, layered saffron nomad dress with armored shoulder pieces and oversized veil, desert observatory carved into a cliff, two moons, dust storm on the horizon, unusual foreshortening, atmospheric anime art"
    },
    {
      "id": "underwater_library",
      "prompt": "masterpiece, best quality, highres, 1girl, solo, adult woman floating horizontally between underwater library shelves, upside-down reading pose with a glowing book, translucent teal diving dress, antique brass breathing apparatus, schools of luminous fish, sunbeams through the ocean surface, readable visual hierarchy, elegant surreal anime illustration"
    },
    {
      "id": "rooftop_kite",
      "prompt": "masterpiece, best quality, highres, 1girl, solo, adult woman crouching in a deep asymmetrical rooftop pose while flying a gigantic paper kite shaped like a koi dragon, layered patchwork rain poncho, climbing harness, orange boots, dense old city rooftops below, monsoon clouds opening to sunset, dynamic diagonal composition, energetic anime illustration"
    },
    {
      "id": "fungal_greenhouse",
      "prompt": "masterpiece, best quality, highres, 1girl, solo, adult woman folded into a compact sitting pose inside a giant bioluminescent mushroom greenhouse, one knee raised and one arm threading glowing vines, iridescent beekeeper jacket, translucent gloves, glass domes and oversized fungi, violet mist, macro-scale environment, unusual color palette, detailed fantasy anime art"
    },
    {
      "id": "ice_carnival",
      "prompt": "masterpiece, best quality, highres, 1girl, solo, adult woman skating backward while holding a frozen parasol, elegant black-and-gold harlequin winter suit with sculptural sleeves, one leg extended in a difficult arabesque, abandoned ice carnival, frozen carousel horses, aurora overhead, reflective ice, dramatic motion trail, polished anime illustration"
    },
    {
      "id": "paper_city",
      "prompt": "masterpiece, best quality, highres, 1girl, solo, adult woman emerging sideways from a folded paper city, reclining pose with one boot pointed toward the viewer, origami architect coat made of layered maps, red scarf turning into birds, impossible paper skyscrapers, warm desk-lamp lighting against deep blue shadows, forced perspective, surreal anime artwork"
    },
    {
      "id": "volcanic_bridge",
      "prompt": "masterpiece, best quality, highres, 1girl, solo, adult woman seated backwards astride a narrow chain bridge above a volcanic crater, looking directly at viewer, ceremonial obsidian armor mixed with flowing coral fabric, braided hair threaded with tiny lanterns, lava waterfalls, smoke and stars, strong silhouette, dangerous scale, cinematic anime fantasy illustration"
    },
    {
      "id": "museum_giant",
      "prompt": "masterpiece, best quality, highres, 1girl, solo, adult woman standing on the palm of a colossal stone statue inside a flooded museum, balancing pose with arms spread, avant-garde cream suit with aquatic fins and red gloves, floating paintings, whale skeleton overhead, shafts of light through the ceiling, scale contrast, poetic surreal anime illustration"
    }
  ],
  "models": [
    {
      "id": "anima_aesthetic_v11",
      "label": "Anima Aesthetic v1.1",
      "kind": "anima",
      "local_name": "anima/anima_aestheticV11.safetensors",
      "source": "https://civitai.com/models/2495369/kirazuri-anima",
      "source_version": "Anima Aesthetic v1.1",
      "sampler": "euler",
      "scheduler": "normal",
      "steps": 40,
      "cfg": 5.0,
      "width": 1024,
      "height": 1536
    },
    {
      "id": "rouwei_v080_epsilon",
      "label": "RouWei v0.8.0 epsilon",
      "kind": "sdxl",
      "local_name": "rouwei_v080Epsilon.safetensors",
      "download_url": "https://civitai.com/api/download/models/1832460",
      "source": "https://civitai.com/models/950531/rouwei?modelVersionId=1832460",
      "source_version": "1832460",
      "sha256": "1ABA15DECD15DA1810054ED4C58984610DFAFC728BEF2FB3F3EBE846B2E287A4",
      "sampler": "euler_ancestral",
      "scheduler": "normal",
      "steps": 40,
      "cfg": 7.0,
      "width": 1024,
      "height": 1536,
      "prediction_type": "epsilon"
    },
    {
      "id": "nova_anime_xl_il_v190",
      "label": "Nova Anime XL IL v19.0",
      "kind": "sdxl",
      "local_name": "novaAnimeXL_ilV190.safetensors",
      "download_url": "https://civitai.com/api/download/models/2940478",
      "source": "https://civitai.com/models/376130",
      "source_version": "2940478",
      "sha256": "FA486CAAFC330F133605D3C18B418D183812F14946631C6544BFB28730DB6D6F",
      "sampler": "euler_ancestral",
      "scheduler": "normal",
      "steps": 36,
      "cfg": 5.0,
      "width": 1024,
      "height": 1536
    },
    {
      "id": "janku_v777",
      "label": "JANKU v7.77",
      "kind": "sdxl",
      "local_name": "JANKUTrainedChenkinNoobai_v777.safetensors",
      "download_url": "https://civitai.com/api/download/models/2786084",
      "source": "https://civitai.com/models/1277670/janku-trained-chenkin-and-noobai-rouwei-illustrious-xl",
      "source_version": "2786084",
      "sha256": "88177D224CE97F60CF3E908F87902F913ED4562BD79D237F84238A4354601EFB",
      "sampler": "euler_ancestral",
      "scheduler": "normal",
      "steps": 40,
      "cfg": 5.0,
      "width": 1024,
      "height": 1536
    }
  ]
}

def download(url, target, headers=None):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>1024: print('exists:',target); return
    h={'Authorization':f'Bearer {HF_TOKEN}'} if 'huggingface.co' in url and HF_TOKEN else {}
    if headers: h.update(headers)
    with requests.get(url,headers=h,stream=True,timeout=60) as r:
        r.raise_for_status()
        with open(target,'wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk: f.write(chunk)
    print('downloaded:',target)
def civitai(url,target):
    if not CIVITAI_API_TOKEN:
        raise RuntimeError('CIVITAI_API_TOKEN is required for Civitai checkpoints.')
    download(url,target,{'Authorization':f'Bearer {CIVITAI_API_TOKEN}'})

# Shared Anima assets (needed for Anima Aesthetic; also reused by workflows)
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors',MODEL_ROOT+'/text_encoders/qwen_3_06b_base.safetensors',{'Authorization':f'Bearer {HF_TOKEN}'})
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors',MODEL_ROOT+'/vae/qwen_image_vae.safetensors',{'Authorization':f'Bearer {HF_TOKEN}'})
download('https://huggingface.co/Comfy-Org/Anima-LLLite/resolve/main/model_patches/anima-lllite-any-test-like-v2.safetensors',MODEL_ROOT+'/model_patches/anima-lllite-any-test-like-v2.safetensors',{'Authorization':f'Bearer {HF_TOKEN}'})

for model in MANIFEST['models']:
    if model['id']=='rouwei_v080_epsilon' and model.get('prediction_type')!='epsilon':
        raise RuntimeError('RouWei must be epsilon, not vpred')
    if model['kind']=='sdxl':
        civitai(model['download_url'], MODEL_ROOT+'/checkpoints/'+model['local_name']); verify_sha256(MODEL_ROOT+'/checkpoints/'+model['local_name'], model.get('sha256','')) if model.get('sha256') else None
    elif model['kind']=='anima':
        # Anima goes to diffusion_models/anima/
        civitai('https://civitai.red/api/download/models/3126581?fileId=3007030', MODEL_ROOT+'/diffusion_models/'+model['local_name'])
    else:
        civitai(model['download_url'], MODEL_ROOT+'/checkpoints/'+model['local_name']); verify_sha256(MODEL_ROOT+'/checkpoints/'+model['local_name'], model.get('sha256','')) if model.get('sha256') else None
print('All 4 models ready:', ', '.join(m['label'] for m in MANIFEST['models']))
# keep MANIFEST in globals for compare cell
globals()['manifest'] = MANIFEST


In [ ]:
# @title 3b) Compare settings — формы (настрой сбоку ДО запуска очереди) ⭐
# Настрой промпты и параметры сэмплинга каждой модели здесь (панель справа), затем запускай ячейку 4.
# Все поля — Colab Forms (#@param): не редактируй код внутри, меняй значения в форме.

# ── Общие ──────────────────────────────────────────────────────────────
COMPARE_MODE = "full"  # @param ["full", "validation"] {allow-input: false}
COMPARE_CLEAN = True  # @param {type:"boolean"}
SEED = 424242  # @param {type:"integer"}

# ── Anima Aesthetic v1.1 ───────────────────────────────────────────────
anima_steps = 30  # @param {type:"slider", min:10, max:100, step:1}
anima_cfg = 4.0  # @param {type:"slider", min:1, max:15, step:0.5}
anima_sampler = "euler"  # @param ["euler", "euler_ancestral", "dpmpp_2m", "dpmpp_sde", "uni_pc"] {allow-input: true}
anima_scheduler = "simple"  # @param ["normal", "karras", "exponential", "sgm_uniform", "simple"] {allow-input: true}
anima_width = 1024  # @param {type:"integer"}
anima_height = 1536  # @param {type:"integer"}

# ── RouWei v0.8 epsilon ────────────────────────────────────────────────
rouwei_steps = 28  # @param {type:"slider", min:10, max:100, step:1}
rouwei_cfg = 5.0  # @param {type:"slider", min:1, max:15, step:0.5}
rouwei_sampler = "euler_ancestral"  # @param ["euler_ancestral", "euler", "dpmpp_2m", "dpmpp_sde", "uni_pc"] {allow-input: true}
rouwei_scheduler = "normal"  # @param ["normal", "karras", "exponential", "sgm_uniform", "simple"] {allow-input: true}
rouwei_width = 1024  # @param {type:"integer"}
rouwei_height = 1536  # @param {type:"integer"}

# ── Nova Anime XL v19 ──────────────────────────────────────────────────
nova_steps = 30  # @param {type:"slider", min:10, max:100, step:1}
nova_cfg = 5.0  # @param {type:"slider", min:1, max:15, step:0.5}
nova_sampler = "euler_ancestral"  # @param ["euler_ancestral", "euler", "dpmpp_2m", "dpmpp_sde", "uni_pc"] {allow-input: true}
nova_scheduler = "normal"  # @param ["normal", "karras", "exponential", "sgm_uniform", "simple"] {allow-input: true}
nova_width = 1024  # @param {type:"integer"}
nova_height = 1536  # @param {type:"integer"}

# ── JANKU v7.77 ────────────────────────────────────────────────────────
janku_steps = 30  # @param {type:"slider", min:10, max:100, step:1}
janku_cfg = 5.0  # @param {type:"slider", min:1, max:15, step:0.5}
janku_sampler = "euler_ancestral"  # @param ["euler_ancestral", "euler", "dpmpp_2m", "dpmpp_sde", "uni_pc"] {allow-input: true}
janku_scheduler = "normal"  # @param ["normal", "karras", "exponential", "sgm_uniform", "simple"] {allow-input: true}
janku_width = 1024  # @param {type:"integer"}
janku_height = 1536  # @param {type:"integer"}

# ── Промпты (10 штук) — редактируй тут, не в ячейке очереди ──────────
negative_prompt = "worst quality, low quality, blurry, bad anatomy, bad hands, extra fingers, watermark, text, signature"  # @param {type:"string"}

prompt_01_gravity_workshop = "masterpiece, best quality, highres, 1girl, solo, adult woman, upside-down suspended pose inside a zero-gravity clockwork workshop, one hand reaching toward a floating brass astrolabe, asymmetrical cobalt flight suit with transparent panels, loose ribbons and tools orbiting around her, impossible perspective, dramatic rim light, intricate mechanical background, cinematic anime illustration"  # @param {type:"string"}
prompt_02_mirror_train = "masterpiece, best quality, highres, 1girl, solo, adult woman balancing in a one-legged dancer pose on the roof of a moving night train, long mirrored coat over sculptural white clothing, windblown silver hair, neon railway reflected in rain, multiple reflections in broken glass panels, wide-angle composition, surreal city lights, highly detailed anime illustration"  # @param {type:"string"}
prompt_03_desert_observatory = "masterpiece, best quality, highres, 1girl, solo, adult woman kneeling sideways on a giant observatory telescope, looking over her shoulder while adjusting a star map, layered saffron nomad dress with armored shoulder pieces and oversized veil, desert observatory carved into a cliff, two moons, dust storm on the horizon, unusual foreshortening, atmospheric anime art"  # @param {type:"string"}
prompt_04_underwater_library = "masterpiece, best quality, highres, 1girl, solo, adult woman floating horizontally between underwater library shelves, upside-down reading pose with a glowing book, translucent teal diving dress, antique brass breathing apparatus, schools of luminous fish, sunbeams through the ocean surface, readable visual hierarchy, elegant surreal anime illustration"  # @param {type:"string"}
prompt_05_rooftop_kite = "masterpiece, best quality, highres, 1girl, solo, adult woman crouching in a deep asymmetrical rooftop pose while flying a gigantic paper kite shaped like a koi dragon, layered patchwork rain poncho, climbing harness, orange boots, dense old city rooftops below, monsoon clouds opening to sunset, dynamic diagonal composition, energetic anime illustration"  # @param {type:"string"}
prompt_06_fungal_greenhouse = "masterpiece, best quality, highres, 1girl, solo, adult woman folded into a compact sitting pose inside a giant bioluminescent mushroom greenhouse, one knee raised and one arm threading glowing vines, iridescent beekeeper jacket, translucent gloves, glass domes and oversized fungi, violet mist, macro-scale environment, unusual color palette, detailed fantasy anime art"  # @param {type:"string"}
prompt_07_ice_carnival = "masterpiece, best quality, highres, 1girl, solo, adult woman skating backward while holding a frozen parasol, elegant black-and-gold harlequin winter suit with sculptural sleeves, one leg extended in a difficult arabesque, abandoned ice carnival, frozen carousel horses, aurora overhead, reflective ice, dramatic motion trail, polished anime illustration"  # @param {type:"string"}
prompt_08_paper_city = "masterpiece, best quality, highres, 1girl, solo, adult woman emerging sideways from a folded paper city, reclining pose with one boot pointed toward the viewer, origami architect coat made of layered maps, red scarf turning into birds, impossible paper skyscrapers, warm desk-lamp lighting against deep blue shadows, forced perspective, surreal anime artwork"  # @param {type:"string"}
prompt_09_volcanic_bridge = "masterpiece, best quality, highres, 1girl, solo, adult woman seated backwards astride a narrow chain bridge above a volcanic crater, looking directly at viewer, ceremonial obsidian armor mixed with flowing coral fabric, braided hair threaded with tiny lanterns, lava waterfalls, smoke and stars, strong silhouette, dangerous scale, cinematic anime fantasy illustration"  # @param {type:"string"}
prompt_10_museum_giant = "masterpiece, best quality, highres, 1girl, solo, adult woman standing on the palm of a colossal stone statue inside a flooded museum, balancing pose with arms spread, avant-garde cream suit with aquatic fins and red gloves, floating paintings, whale skeleton overhead, shafts of light through the ceiling, scale contrast, poetic surreal anime illustration"  # @param {type:"string"}

# Применяем формы к MANIFEST (переопределяет значения из ячейки 3, если она уже выполнена)
try:
    _forms_prompts = [
        ("gravity_workshop", prompt_01_gravity_workshop),
        ("mirror_train", prompt_02_mirror_train),
        ("desert_observatory", prompt_03_desert_observatory),
        ("underwater_library", prompt_04_underwater_library),
        ("rooftop_kite", prompt_05_rooftop_kite),
        ("fungal_greenhouse", prompt_06_fungal_greenhouse),
        ("ice_carnival", prompt_07_ice_carnival),
        ("paper_city", prompt_08_paper_city),
        ("volcanic_bridge", prompt_09_volcanic_bridge),
        ("museum_giant", prompt_10_museum_giant),
    ]
    if "MANIFEST" in globals():
        MANIFEST["seed"] = int(SEED)
        MANIFEST["negative_prompt"] = negative_prompt
        MANIFEST["prompts"] = [{"id": pid, "prompt": txt} for pid, txt in _forms_prompts if txt.strip()]
        # map form values to models
        _overrides = {
            "anima_aesthetic_v11": (anima_steps, anima_cfg, anima_sampler, anima_scheduler, anima_width, anima_height),
            "rouwei_v080_epsilon": (rouwei_steps, rouwei_cfg, rouwei_sampler, rouwei_scheduler, rouwei_width, rouwei_height),
            "nova_anime_xl_il_v190": (nova_steps, nova_cfg, nova_sampler, nova_scheduler, nova_width, nova_height),
            "janku_v777": (janku_steps, janku_cfg, janku_sampler, janku_scheduler, janku_width, janku_height),
        }
        for m in MANIFEST["models"]:
            if m["id"] in _overrides:
                st, cf, sm, sc, w, h = _overrides[m["id"]]
                m["steps"] = int(st); m["cfg"] = float(cf); m["sampler"] = sm; m["scheduler"] = sc; m["width"] = int(w); m["height"] = int(h)
        os.environ["COMPARE_MODE"] = COMPARE_MODE
        os.environ["COMPARE_CLEAN"] = "1" if COMPARE_CLEAN else "0"
        print(f"✓ Forms применены: {len(MANIFEST['prompts'])} промптов, seed={SEED}, mode={COMPARE_MODE}")
        for m in MANIFEST["models"]:
            print(f"  {m['label']}: {m['steps']} steps, cfg {m['cfg']}, {m['sampler']}/{m['scheduler']}, {m['width']}x{m['height']}")
    else:
        print("⚠ MANIFEST ещё не создан — сначала запусти ячейку 3 (Download), затем вернись сюда.")
except Exception as e:
    print(f"⚠ Ошибка применения форм: {e}")
    raise



In [ ]:
# @title 5) Launch ComfyUI + Cloudflare Quick Tunnel
# Реализация туннеля идентична Hermes Dashboard-ячейке:
#   1. поднимаем ComfyUI локально и ждём готовности;
#   2. находим/скачиваем cloudflared;
#   3. останавливаем старый туннель при повторном запуске;
#   4. поднимаем Quick Tunnel на локальный порт ComfyUI;
#   5. читаем URL из лога, ждём DNS, проверяем публичный доступ.
import base64, os, re, shutil, socket, stat, subprocess, threading, time
import queue
from pathlib import Path

import requests

LOW_VRAM_STABLE = False  # False: --lowvram + Dynamic VRAM (optimal for T4 15GB); True: --novram ultra-low (2.5x slower, risks RAM OOM)
COMFY_ROOT = Path(globals().get("COMFY_ROOT", "/content/ComfyUI"))
OUTPUT_DIR = COMFY_ROOT / "output"
COMFY_PORT = globals().get("COMFY_PORT", 8188)

# ── Model Manager token bridge (keys live in its private.key pickle) ────────
MODEL_MANAGER_DIR = COMFY_ROOT / "custom_nodes" / "ComfyUI-Model-Manager"
if MODEL_MANAGER_DIR.exists():
    import pickle

    manager_key_file = MODEL_MANAGER_DIR / "private.key"
    manager_keys = {}
    if manager_key_file.exists():
        try:
            with manager_key_file.open("rb") as stream:
                loaded = pickle.load(stream)
            if isinstance(loaded, dict):
                manager_keys.update(loaded)
        except Exception:
            print("Existing Model Manager key file was unreadable; recreating it.")

    hf_for_manager = (os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN") or "").strip()
    civitai_for_manager = os.environ.get("CIVITAI_API_TOKEN", "").strip()
    if hf_for_manager:
        manager_keys["huggingface"] = hf_for_manager
    if civitai_for_manager:
        manager_keys["civitai"] = civitai_for_manager
    if manager_keys:
        manager_key_tmp = manager_key_file.with_suffix(".private.key.tmp")
        with manager_key_tmp.open("wb") as stream:
            pickle.dump(manager_keys, stream, protocol=pickle.HIGHEST_PROTOCOL)
        manager_key_tmp.replace(manager_key_file)
    print(
        "Model Manager token bridge:",
        "HF=" + ("yes" if hf_for_manager else "existing/none"),
        "Civitai=" + ("yes" if civitai_for_manager else "existing/none"),
    )
else:
    print("⚠ ComfyUI-Model-Manager not found — run the install cell first.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def stop_process(proc):
    if proc is None or proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()


# Safe rerun: stop processes started by any earlier launch-cell version.
for process_name in ("_TUNNEL_PROC", "_COMFY_PROC"):
    stop_process(globals().get(process_name))
old_log = globals().get("_COMFY_LOG")
if old_log is not None:
    try:
        old_log.close()
    except Exception:
        pass


def port_open(host, port, timeout=1):
    s = socket.socket()
    s.settimeout(timeout)
    try:
        return s.connect_ex((host, port)) == 0
    finally:
        s.close()


# ══════════════════════════════════════════════════════════════════════════
# 1. ЗАПУСК COMFYUI ЛОКАЛЬНО
# ══════════════════════════════════════════════════════════════════════════
comfy_args = [
    "python", "main.py", "--listen", "0.0.0.0", "--port", str(COMFY_PORT),
    "--enable-cors-header", "*", "--output-directory", str(OUTPUT_DIR),
]
comfy_args += (
    ["--novram", "--disable-smart-memory", "--cache-none", "--force-upcast-attention"]
    if LOW_VRAM_STABLE else ["--lowvram", "--preview-method", "auto"]
)
_COMFY_LOG = open("/content/comfyui.log", "a", encoding="utf-8", buffering=1)
_COMFY_PROC = subprocess.Popen(
    comfy_args, cwd=COMFY_ROOT, stdout=_COMFY_LOG, stderr=subprocess.STDOUT,
)


def local_comfy_ready(timeout=300):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if _COMFY_PROC.poll() is not None:
            raise RuntimeError("ComfyUI exited. Inspect /content/comfyui.log")
        try:
            response = requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=5)
            if response.ok:
                return True
        except requests.RequestException:
            time.sleep(2)
    return False


if not local_comfy_ready():
    raise TimeoutError("ComfyUI did not become ready within 300 seconds.")
print("✓ ComfyUI is ready locally on port", COMFY_PORT)


# ══════════════════════════════════════════════════════════════════════════
# 2. НАХОДИМ / СКАЧИВАЕМ CLOUDFLARED  (как в Hermes Dashboard-ячейке)
# ══════════════════════════════════════════════════════════════════════════
cloudflared_candidates = [
    shutil.which("cloudflared"),
    "/usr/local/bin/cloudflared",
]

CLOUDFLARED_BIN = None

for c in cloudflared_candidates:
    if c and Path(c).exists():
        CLOUDFLARED_BIN = Path(c)
        break

if CLOUDFLARED_BIN is None:
    machine = os.uname().machine.lower()

    if machine in ("x86_64", "amd64"):
        cf_arch = "amd64"
    elif machine in ("aarch64", "arm64"):
        cf_arch = "arm64"
    else:
        raise RuntimeError(f"Unknown architecture: {machine}")

    CLOUDFLARED_BIN = Path("/tmp/cloudflared-comfy")
    download_url = (
        "https://github.com/cloudflare/cloudflared/"
        f"releases/latest/download/cloudflared-linux-{cf_arch}"
    )

    print("Скачиваю cloudflared...")
    subprocess.run(
        ["curl", "-fL", "--retry", "3", download_url, "-o", str(CLOUDFLARED_BIN)],
        check=True,
    )
    CLOUDFLARED_BIN.chmod(CLOUDFLARED_BIN.stat().st_mode | stat.S_IXUSR)

print("cloudflared:", CLOUDFLARED_BIN)


# ══════════════════════════════════════════════════════════════════════════
# 3. ПОВТОРНЫЙ ЗАПУСК — ОСТАНАВЛИВАЕМ СТАРЫЙ ТУННЕЛЬ
# ══════════════════════════════════════════════════════════════════════════
old_cf = globals().get("_CLOUDFLARED_PROC")
if old_cf is not None:
    try:
        if old_cf.poll() is None:
            print("Останавливаю предыдущий tunnel...")
            old_cf.terminate()
            try:
                old_cf.wait(timeout=10)
            except Exception:
                old_cf.kill()
    except Exception:
        pass


# ══════════════════════════════════════════════════════════════════════════
# 4. ПОДНИМАЕМ QUICK TUNNEL НА COMFYUI
# ══════════════════════════════════════════════════════════════════════════
print("=" * 78)
print("ЗАПУСК CLOUDFLARE QUICK TUNNEL -> COMFYUI")
print("=" * 78)

_CLOUDFLARED_PROC = subprocess.Popen(
    [
        str(CLOUDFLARED_BIN),
        "tunnel",
        "--no-autoupdate",
        "--url",
        f"http://127.0.0.1:{COMFY_PORT}",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)


# ══════════════════════════════════════════════════════════════════════════
# 5. ЧИТАЕМ URL ИЗ ЛОГА
# ══════════════════════════════════════════════════════════════════════════
TUNNEL_URL = None

cf_lines = []

pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

deadline = time.time() + 90

while time.time() < deadline:

    if _CLOUDFLARED_PROC.poll() is not None:
        break

    line = _CLOUDFLARED_PROC.stdout.readline()

    if line:
        cf_lines.append(line.rstrip())
        print(line, end="")

        match = pattern.search(line)

        if match:
            TUNNEL_URL = match.group(0)
            break

    else:
        time.sleep(0.2)

if not TUNNEL_URL:
    print()
    print("=== CLOUDFLARED OUTPUT ===")
    print("\n".join(cf_lines[-200:]))

    # Fallback hint: the next cell exposes ComfyUI through bore.pub instead.
    raise RuntimeError(
        "Не удалось получить Quick Tunnel URL. "
        "Если сеть блокирует trycloudflare — запусти следующую ячейку (bore.pub fallback)."
    )

print()
print("✓ ComfyUI Tunnel:", TUNNEL_URL)


# ══════════════════════════════════════════════════════════════════════════
# 6. ЖДЁМ DNS
# ══════════════════════════════════════════════════════════════════════════
TUNNEL_HOST = TUNNEL_URL.replace("https://", "").split("/")[0]

dns_ok = False

for attempt in range(1, 31):
    try:
        infos = socket.getaddrinfo(TUNNEL_HOST, 443)
        if infos:
            dns_ok = True
            break
    except socket.gaierror:
        pass
    time.sleep(2)

print("DNS:", "OK" if dns_ok else "NOT READY")


# ══════════════════════════════════════════════════════════════════════════
# 7. ПРОВЕРЯЕМ ПУБЛИЧНЫЙ ДОСТУП
# ══════════════════════════════════════════════════════════════════════════
if dns_ok:

    print("=" * 78)
    print("PUBLIC COMFYUI TEST")
    print("=" * 78)

    public_test = subprocess.run(
        [
            "curl", "-sS", "-L",
            "--connect-timeout", "15",
            "--max-time", "60",
            "--retry", "5",
            "--retry-delay", "2",
            "-o", "/dev/null",
            "-w", "%{http_code}",
            TUNNEL_URL + "/system_stats",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        timeout=90,
    )

    public_http = public_test.stdout.strip()
    print("Public /system_stats HTTP:", public_http)

    if public_test.stderr:
        print("curl stderr:", public_test.stderr[:3000])

    if public_http not in ("200", "403"):
        print("⚠ Public check inconclusive — tunnel может ещё прогреваться.")

print()
print("=" * 78)
print("✓ COMFYUI ГОТОВ — ОТКРЫВАЙ В БРАУЗЕРЕ:")
print(TUNNEL_URL)
print("=" * 78)
print("Если ссылка даёт 403 — запусти следующую ячейку (bore.pub fallback).")

globals()["TUNNEL_URL"] = TUNNEL_URL

In [ ]:
# @title 5a) Watchdog: ComfyUI + Cloudflare Tunnel keepalive
# Keeps BOTH services alive. ComfyUI-Manager reboot is given a grace period:
# if Manager already spawned/replaced main.py, watchdog detects that process/port
# and never starts a duplicate.
import os, re, shutil, socket, subprocess, time
from pathlib import Path
import requests

COMFY_PORT = int(globals().get("COMFY_PORT", 8188))
COMFY_ROOT = Path(globals().get("COMFY_ROOT", "/content/ComfyUI"))
COMFY_RESTART_GRACE = 25
COMFY_HUNG_GRACE = 120
CLOUDFLARED_BIN = globals().get("CLOUDFLARED_BIN")
if CLOUDFLARED_BIN is None:
    for candidate in [shutil.which("cloudflared"), "/usr/local/bin/cloudflared", "/tmp/cloudflared-comfy"]:
        if candidate and Path(candidate).exists():
            CLOUDFLARED_BIN = Path(candidate)
            break
if CLOUDFLARED_BIN is None:
    raise RuntimeError("cloudflared not found — run the launch/tunnel cell first.")

_cf_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

def _wd_health():
    try:
        return requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=4).ok
    except Exception:
        return False

def _wd_main_pids():
    # Detect a Manager-started replacement before spawning our own process.
    try:
        out = subprocess.check_output(
            ["pgrep", "-f", rf"python.*main\.py.*--port[ =]{COMFY_PORT}"],
            text=True, stderr=subprocess.DEVNULL,
        )
        return [int(x) for x in out.split() if x.isdigit()]
    except Exception:
        return []

def _wd_comfy_args():
    args = globals().get("comfy_args")
    if isinstance(args, (list, tuple)) and args:
        return list(args)
    output_dir = COMFY_ROOT / "output"
    low_stable = bool(globals().get("LOW_VRAM_STABLE", False))
    args = [
        "python", "main.py", "--listen", "0.0.0.0", "--port", str(COMFY_PORT),
        "--enable-cors-header", "*", "--output-directory", str(output_dir),
    ]
    args += (
        ["--novram", "--disable-smart-memory", "--cache-none", "--force-upcast-attention"]
        if low_stable else ["--lowvram", "--preview-method", "auto"]
    )
    globals()["comfy_args"] = args
    return args

def _wd_spawn_comfy():
    # Final race check: Manager may have completed reboot while we were waiting.
    if _wd_health() or _wd_main_pids():
        return False
    old_log = globals().get("_COMFY_LOG")
    try:
        if old_log is not None:
            old_log.close()
    except Exception:
        pass
    log = open("/content/comfyui.log", "a", encoding="utf-8", buffering=1)
    proc = subprocess.Popen(
        _wd_comfy_args(), cwd=COMFY_ROOT, stdout=log, stderr=subprocess.STDOUT
    )
    globals()["_COMFY_LOG"] = log
    globals()["_COMFY_PROC"] = proc
    globals()["_COMFY_DOWN_SINCE"] = time.time()
    print(f"[{time.strftime('%H:%M:%S')}] ComfyUI watchdog spawned PID {proc.pid}.")
    return True

def _ensure_comfy_running():
    if _wd_health():
        globals()["_COMFY_DOWN_SINCE"] = None
        return True

    now = time.time()
    down_since = globals().get("_COMFY_DOWN_SINCE")
    if not down_since:
        down_since = now
        globals()["_COMFY_DOWN_SINCE"] = down_since

    proc = globals().get("_COMFY_PROC")
    proc_alive = proc is not None and proc.poll() is None
    pids = _wd_main_pids()

    # Most important Manager-compat rule: if any replacement main.py exists,
    # do not race it. The same Cloudflare tunnel already targets localhost:port.
    if pids and (not proc_alive or proc.pid not in pids):
        if int(now - down_since) % 30 < 5:
            print(f"[{time.strftime('%H:%M:%S')}] Manager/external ComfyUI startup detected (PIDs {pids}); waiting for port.")
        return False

    grace = COMFY_HUNG_GRACE if proc_alive else COMFY_RESTART_GRACE
    if now - down_since < grace:
        return False

    if proc_alive:
        print(f"[{time.strftime('%H:%M:%S')}] ComfyUI owned PID {proc.pid} is unhealthy for {grace}s; restarting it.")
        try:
            proc.terminate()
            proc.wait(timeout=10)
        except Exception:
            try:
                proc.kill()
            except Exception:
                pass
        time.sleep(2)

    if _wd_health() or _wd_main_pids():
        return _wd_health()
    _wd_spawn_comfy()

    deadline = time.time() + 300
    while time.time() < deadline:
        if _wd_health():
            globals()["_COMFY_DOWN_SINCE"] = None
            print(f"[{time.strftime('%H:%M:%S')}] ✓ ComfyUI recovered; existing tunnel remains attached to port {COMFY_PORT}.")
            return True
        proc = globals().get("_COMFY_PROC")
        if proc is not None and proc.poll() is not None:
            print(f"[{time.strftime('%H:%M:%S')}] ComfyUI restart exited={proc.poll()}; retry will happen on next watchdog cycle.")
            break
        time.sleep(3)
    return False

def _restart_cf():
    old = globals().get("_CLOUDFLARED_PROC")
    if old is not None and old.poll() is None:
        try:
            old.terminate()
            old.wait(timeout=5)
        except Exception:
            try: old.kill()
            except Exception: pass
    print(f"[{time.strftime('%H:%M:%S')}] Restarting cloudflared -> 127.0.0.1:{COMFY_PORT} ...")
    proc = subprocess.Popen(
        [str(CLOUDFLARED_BIN), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{COMFY_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    globals()["_CLOUDFLARED_PROC"] = proc
    url = None
    deadline = time.time() + 90
    lines = []
    while time.time() < deadline:
        if proc.poll() is not None:
            break
        line = proc.stdout.readline() if proc.stdout else ""
        if line:
            lines.append(line.rstrip())
            print(line, end="")
            match = _cf_pattern.search(line)
            if match:
                url = match.group(0)
                break
        else:
            time.sleep(0.2)
    if not url:
        print("\n".join(lines[-20:]) if lines else "cloudflared produced no URL")
        return None
    globals()["TUNNEL_URL"] = url
    print(f"✓ TUNNEL_URL: {url}")
    return url

_ensure_comfy_running()
cf = globals().get("_CLOUDFLARED_PROC")
if cf is None or cf.poll() is not None:
    _restart_cf()

print("Unified ComfyUI + Cloudflare watchdog started. Interrupt this cell to stop monitoring.")
_tick = 0
try:
    while True:
        time.sleep(5)
        _tick += 5
        local_ok = _ensure_comfy_running()
        cf = globals().get("_CLOUDFLARED_PROC")
        if cf is None or cf.poll() is not None:
            _restart_cf()
            _tick = 0
            continue
        if _tick >= 60:
            _tick = 0
            url = globals().get("TUNNEL_URL", "")
            if local_ok and url:
                try:
                    public = requests.get(url + "/system_stats", timeout=8)
                    print(f"[{time.strftime('%H:%M:%S')}] keepalive local=OK public={public.status_code} {url}")
                    if public.status_code >= 500:
                        _restart_cf()
                except Exception as exc:
                    print(f"[{time.strftime('%H:%M:%S')}] public ping failed ({exc}); restarting tunnel only.")
                    _restart_cf()
            else:
                print(f"[{time.strftime('%H:%M:%S')}] keepalive local={'OK' if local_ok else 'RECOVERING'}")
except KeyboardInterrupt:
    print("Watchdog stopped by user.")


In [ ]:
# @title 4) Queue a reproducible ComfyUI compare
import json, os, shutil, time, requests
from pathlib import Path
COMPARE_OUTPUT = COMFY_ROOT / 'output' / 'compare'
COMPARE_OUTPUT = Path(COMPARE_OUTPUT)
if os.environ.get('COMPARE_CLEAN','1')!='0' and COMPARE_OUTPUT.exists():
    shutil.rmtree(COMPARE_OUTPUT)
COMPARE_OUTPUT.mkdir(parents=True, exist_ok=True)
API = 'http://127.0.0.1:8188'
# VRAM check for compare (4 models sequential, but RAM cache matters)
def _detect_vram(default=14.0):
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_properties(0).total_memory/(1024**3)
    except: pass
    return default
_vtotal=_detect_vram(); _vbudget=_vtotal*0.7
print(f"VRAM ~{_vtotal:.1f}GB | 70% ~{_vbudget:.1f}GB — compare runs 40 jobs sequentially, LOW_VRAM_STABLE=False OK на T4")
if _vbudget<7: print("⚠ Low budget — совет: COMPARE_MODE=validation (1 промпт×4) для теста")


def graph_for(model, case, case_index):
    seed = MANIFEST['seed'] + case_index
    common = {
        '2': {'class_type':'CLIPTextEncode','inputs':{'text':case['prompt'],'clip':['1',1]}},
        '3': {'class_type':'CLIPTextEncode','inputs':{'text':case.get('negative_prompt',MANIFEST['negative_prompt']),'clip':['1',1]}},
        '4': {'class_type':'EmptyLatentImage','inputs':{'width':model['width'],'height':model['height'],'batch_size':1}},
        '5': {'class_type':'KSampler','inputs':{'seed':seed,'steps':model['steps'],'cfg':model['cfg'],'sampler_name':model['sampler'],'scheduler':model['scheduler'],'denoise':1.0,'model':['0',0],'positive':['2',0],'negative':['3',0],'latent_image':['4',0]}},
        '6': {'class_type':'VAEDecode','inputs':{'samples':['5',0],'vae':['7',0]}},
        '8': {'class_type':'SaveImage','inputs':{'filename_prefix':'compare/'+case['id']+'/'+model['id'],'images':['6',0]}},
    }
    if model['kind']=='anima':
        common.update({'0':{'class_type':'UNETLoader','inputs':{'unet_name':model['local_name'],'weight_dtype':'default'}},'1':{'class_type':'CLIPLoader','inputs':{'clip_name':'qwen_3_06b_base.safetensors','type':'stable_diffusion','device':'default'}},'7':{'class_type':'VAELoader','inputs':{'vae_name':'qwen_image_vae.safetensors'}}})
        common['2']['inputs']['clip']=['1',0]; common['3']['inputs']['clip']=['1',0]
    else:
        common.update({'0':{'class_type':'CheckpointLoaderSimple','inputs':{'ckpt_name':model['local_name']}}})
        common['2']['inputs']['clip']=['0',1]; common['3']['inputs']['clip']=['0',1]; common['6']['inputs']['vae']=['0',2]
    return common

def queue_graph(model, case, case_index):
    payload={'prompt':graph_for(model, case, case_index),'client_id':'private-model-compare'}
    response=requests.post(API+'/prompt',json=payload,timeout=30); response.raise_for_status(); return response.json()['prompt_id']

results=[]
cases = MANIFEST.get('prompts') or [{'id':'baseline','prompt':MANIFEST['prompt']}]
if os.environ.get('COMPARE_MODE','').strip().lower() == 'validation':
    cases = cases[:1]
    print('Validation mode: one prompt per model')
for case_index, case in enumerate(cases):
  for model in MANIFEST['models']:
    print('queue:', case['id'], model['label']); pid=queue_graph(model, case, case_index); deadline=time.time()+1800
    while time.time()<deadline:
        history=requests.get(API+'/history/'+pid,timeout=30).json()
        if pid in history:
            results.append({'case':case['id'],'model':model['id'],'label':model['label'],'prompt_id':pid,'history':history[pid]}); print('done:',case['id'],model['label']); break
        time.sleep(5)
    else: raise TimeoutError('Timed out waiting for '+model['label'])
(COMPARE_OUTPUT/'run_metadata.json').write_text(json.dumps({'manifest':MANIFEST,'results':results},ensure_ascii=False,indent=2),encoding='utf-8')

# ── Архив результатов + автоскачивание ────────────────────────────────
import zipfile
zip_path = COMPARE_OUTPUT.parent / "compare.zip"
print(f"Собираю архив {zip_path} ...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in COMPARE_OUTPUT.rglob("*"):
        if fp.is_file():
            zf.write(fp, fp.relative_to(COMPARE_OUTPUT.parent))
    # also include run_metadata
    if (COMPARE_OUTPUT / "run_metadata.json").exists():
        pass  # already included via rglob
print(f"✓ Архив готов: {zip_path} ({zip_path.stat().st_size/1024/1024:.1f} MB, {len(list(COMPARE_OUTPUT.rglob('*')))} файлов)")

# Colab auto-download (best effort)
try:
    from google.colab import files as _colab_files
    print("Скачиваю compare.zip через google.colab.files.download ...")
    _colab_files.download(str(zip_path))
except Exception as e:
    print(f"Colab download пропущен ({e}) — используй FileLink ниже.")
try:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))
except Exception:
    pass
print(f"Архив также доступен по пути: {zip_path}")

print('Compare finished:', COMPARE_OUTPUT)



In [ ]:
# @title 3b) Download YOLO bbox detectors for Face/Hand Detailer (auto)
import os
BBOX_DIR = '/content/ComfyUI/models/ultralytics/bbox'
os.makedirs(BBOX_DIR, exist_ok=True)
def dl_bib(url, out):
    if os.path.exists(out) and os.path.getsize(out) > 0:
        print('Already exists:', out)
        return
    print('Downloading bbox:', os.path.basename(out))
    hf_token = os.environ.get('HUGGINGFACE_TOKEN') or os.environ.get('HF_TOKEN')
    hdr = f'--header="Authorization: Bearer {hf_token}"' if hf_token and 'huggingface.co' in url else ''
    # aria2c fast, fallback to curl
    import subprocess
    cmd = f'aria2c --console-log-level=error -c -x 8 -s 8 -k 1M {hdr} "{url}" -d "{BBOX_DIR}" -o "{os.path.basename(out)}"' if hdr else f'aria2c --console-log-level=error -c -x 8 -s 8 -k 1M "{url}" -d "{BBOX_DIR}" -o "{os.path.basename(out)}"'
    rc = subprocess.run(cmd, shell=True).returncode
    if rc != 0 or not os.path.exists(out):
        print('aria2c failed, trying curl')
        import urllib.request
        try:
            urllib.request.urlretrieve(url, out)
            print('curl OK')
        except Exception as e:
            print('download failed', e)
dl_bib('https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt', f'{BBOX_DIR}/face_yolov8m.pt')
dl_bib('https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8n.pt', f'{BBOX_DIR}/hand_yolov8n.pt')
print('YOLO bbox ready:', os.listdir(BBOX_DIR))


In [ ]:
# @title 5b) Fallback: bore.pub tunnel (если Cloudflare URL заблокирован / 403)
# Некоторые сети/страны получают 403 от Cloudflare на *.trycloudflare.com ссылки.
# Эта ячейка открывает ТОТ ЖЕ локальный ComfyUI через bore.pub.
# Обычный HTTP без шифрования и случайный порт — используй только как fallback.
# Останавливает CF-туннель, чтобы не держать два сразу.
import os
import re as _re
import subprocess
import threading as _threading
import time

import requests

COMFY_PORT = globals().get("COMFY_PORT", 8188)

# Останавливаем Cloudflare туннель из предыдущей ячейки (если был).
_old_cf = globals().get("_CLOUDFLARED_PROC")
if _old_cf is not None and _old_cf.poll() is None:
    _old_cf.terminate()
    try:
        _old_cf.wait(timeout=10)
    except Exception:
        _old_cf.kill()
    print("Cloudflare tunnel остановлен.")


def _ensure_bore():
    import shutil

    if shutil.which("bore"):
        return "bore"
    local = "/tmp/bore"
    if os.path.exists(local) and os.access(local, os.X_OK):
        return local
    arch = {"x86_64": "x86_64", "amd64": "x86_64", "aarch64": "aarch64"}.get(
        os.uname().machine.lower(), "x86_64"
    )
    url = (
        "https://github.com/ekzhang/bore/releases/download/v0.5.0/"
        f"bore-v0.5.0-{arch}-unknown-linux-musl.tar.gz"
    )
    print("Скачиваю bore CLI...")
    subprocess.run(["curl", "-fsSL", "-o", "/tmp/bore.tar.gz", url], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/bore.tar.gz", "-C", "/tmp"], check=True)
    os.chmod("/tmp/bore", 0o755)
    return local


BORE_BIN = _ensure_bore()

# Повторный запуск ячейки — останавливаем старый fallback-туннель.
old = globals().get("_BORE_PROC")
if old is not None and old.poll() is None:
    old.terminate()
    try:
        old.wait(timeout=5)
    except Exception:
        old.kill()


# Убеждаемся, что ComfyUI ещё жив (запущен предыдущей ячейкой).
assert globals().get("_COMFY_PROC") is not None and _COMFY_PROC.poll() is None, (
    "ComfyUI не запущен — сначала выполни предыдущую ячейку."
)

_BORE_LOG = open("/content/bore.log", "a", encoding="utf-8", buffering=1)
_BORE_PROC = subprocess.Popen(
    [BORE_BIN, "local", str(COMFY_PORT), "--to", "bore.pub"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

_public_port = None
_deadline = time.time() + 45
_lines = []


def _pump(pipe):
    # Robust: pipe may be None (if Popen failed) or closed; avoid AttributeError
    if pipe is None:
        _lines.append("[bore] no stdout pipe (Popen failed)")
        return
    try:
        for line in iter(pipe.readline, ""):
            if line is None:
                break
            _lines.append(line.rstrip())
            if not line:
                time.sleep(0.1)
    except Exception as e:
        _lines.append(f"[bore pump error] {e}")


_threading.Thread(target=_pump, args=(_BORE_PROC.stdout,), daemon=True).start()

while time.time() < _deadline and _public_port is None:
    if _BORE_PROC.poll() is not None:
        print("bore процесс завершился преждевременно. Лог:")
        print("\n".join(_lines[-20:]))
        break
    for line in list(_lines):
        m = _re.search(r"listening at bore\.pub:(\d+)", line)
        if m:
            _public_port = m.group(1)
            break
    time.sleep(0.5)

if not _public_port:
    print("\n".join(_lines[-20:]) if _lines else "(нет вывода от bore)")
    raise RuntimeError("bore.pub tunnel failed to start. Попробуй перезапустить ячейку или используй Cloudflare tunnel (предыдущая ячейка).")

PUBLIC_URL = f"http://bore.pub:{_public_port}"
print("=" * 78)
print("✓ COMFYUI ЧЕРЕЗ BORE.PUB:", PUBLIC_URL)
print("(plain HTTP — используй только если Cloudflare URL заблокирован)")
print("=" * 78)

for _ in range(6):
    try:
        if requests.get(PUBLIC_URL + "/system_stats", timeout=8).ok:
            print("Public check: OK")
            break
    except requests.RequestException:
        pass
    time.sleep(3)

globals()["PUBLIC_URL"] = PUBLIC_URL


In [ ]:
# @title 5c) Watchdog: ComfyUI + bore.pub keepalive
# Same ComfyUI supervision rules as Cloudflare watchdog; safe with ComfyUI-Manager reboot.
import os, re, subprocess, time
from pathlib import Path
import requests

COMFY_PORT = int(globals().get("COMFY_PORT", 8188))
COMFY_ROOT = Path(globals().get("COMFY_ROOT", "/content/ComfyUI"))
COMFY_RESTART_GRACE = 25
COMFY_HUNG_GRACE = 120
BORE_BIN = globals().get("BORE_BIN")
if BORE_BIN is None:
    import shutil
    BORE_BIN = shutil.which("bore") or ("/tmp/bore" if Path("/tmp/bore").exists() else None)
if BORE_BIN is None:
    raise RuntimeError("bore not found — run the bore fallback cell first.")

def _wd_health():
    try:
        return requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=4).ok
    except Exception:
        return False

def _wd_main_pids():
    try:
        out = subprocess.check_output(["pgrep", "-f", rf"python.*main\.py.*--port[ =]{COMFY_PORT}"], text=True, stderr=subprocess.DEVNULL)
        return [int(x) for x in out.split() if x.isdigit()]
    except Exception:
        return []

def _wd_comfy_args():
    args = globals().get("comfy_args")
    if isinstance(args, (list, tuple)) and args:
        return list(args)
    output_dir = COMFY_ROOT / "output"
    low_stable = bool(globals().get("LOW_VRAM_STABLE", False))
    args = ["python", "main.py", "--listen", "0.0.0.0", "--port", str(COMFY_PORT), "--enable-cors-header", "*", "--output-directory", str(output_dir)]
    args += (["--novram", "--disable-smart-memory", "--cache-none", "--force-upcast-attention"] if low_stable else ["--lowvram", "--preview-method", "auto"])
    globals()["comfy_args"] = args
    return args

def _ensure_comfy_running():
    if _wd_health():
        globals()["_COMFY_DOWN_SINCE"] = None
        return True
    now = time.time()
    down = globals().get("_COMFY_DOWN_SINCE") or now
    globals()["_COMFY_DOWN_SINCE"] = down
    proc = globals().get("_COMFY_PROC")
    alive = proc is not None and proc.poll() is None
    pids = _wd_main_pids()
    if pids and (not alive or proc.pid not in pids):
        return False
    grace = COMFY_HUNG_GRACE if alive else COMFY_RESTART_GRACE
    if now - down < grace:
        return False
    if alive:
        try:
            proc.terminate(); proc.wait(timeout=10)
        except Exception:
            try: proc.kill()
            except Exception: pass
        time.sleep(2)
    if _wd_health() or _wd_main_pids():
        return _wd_health()
    log = open("/content/comfyui.log", "a", encoding="utf-8", buffering=1)
    proc = subprocess.Popen(_wd_comfy_args(), cwd=COMFY_ROOT, stdout=log, stderr=subprocess.STDOUT)
    globals()["_COMFY_LOG"] = log
    globals()["_COMFY_PROC"] = proc
    print(f"[{time.strftime('%H:%M:%S')}] ComfyUI watchdog spawned PID {proc.pid}.")
    deadline = time.time() + 300
    while time.time() < deadline:
        if _wd_health():
            globals()["_COMFY_DOWN_SINCE"] = None
            print("✓ ComfyUI recovered; bore remains attached to the same localhost port.")
            return True
        if proc.poll() is not None:
            break
        time.sleep(3)
    return False

def _restart_bore():
    old = globals().get("_BORE_PROC")
    if old is not None and old.poll() is None:
        try:
            old.terminate(); old.wait(timeout=5)
        except Exception:
            try: old.kill()
            except Exception: pass
    proc = subprocess.Popen([str(BORE_BIN), "local", str(COMFY_PORT), "--to", "bore.pub"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    globals()["_BORE_PROC"] = proc
    lines = []
    deadline = time.time() + 45
    port = None
    while time.time() < deadline and port is None:
        if proc.poll() is not None:
            break
        line = proc.stdout.readline() if proc.stdout else ""
        if line:
            lines.append(line.rstrip())
            m = re.search(r"listening at bore\.pub:(\d+)", line)
            if m: port = m.group(1)
        else:
            time.sleep(0.2)
    if not port:
        print("\n".join(lines[-20:]) if lines else "bore produced no public port")
        return None
    url = f"http://bore.pub:{port}"
    globals()["PUBLIC_URL"] = url
    print("✓ PUBLIC_URL:", url)
    return url

_ensure_comfy_running()
bore = globals().get("_BORE_PROC")
if bore is None or bore.poll() is not None:
    _restart_bore()

print("Unified ComfyUI + bore watchdog started. Interrupt this cell to stop monitoring.")
_tick = 0
try:
    while True:
        time.sleep(5)
        _tick += 5
        local_ok = _ensure_comfy_running()
        bore = globals().get("_BORE_PROC")
        if bore is None or bore.poll() is not None:
            _restart_bore(); _tick = 0; continue
        if _tick >= 60:
            _tick = 0
            url = globals().get("PUBLIC_URL", "")
            if local_ok and url:
                try:
                    public = requests.get(url + "/system_stats", timeout=8)
                    print(f"[{time.strftime('%H:%M:%S')}] keepalive local=OK public={public.status_code} {url}")
                except Exception as exc:
                    print(f"[{time.strftime('%H:%M:%S')}] bore public ping failed ({exc}); restarting tunnel only.")
                    _restart_bore()
            else:
                print(f"[{time.strftime('%H:%M:%S')}] keepalive local={'OK' if local_ok else 'RECOVERING'}")
except KeyboardInterrupt:
    print("Watchdog stopped by user.")
